# Train the Hangman BiLSTM + Attention + Game-State Features on Kaggle GPU

Clones the `approach/bilstm-attention-features` branch and runs training
there. Forked from `approach/bilstm-attention` with one addition: the
model is no longer fed only the board pattern -- it also sees which
letters are confirmed wrong (guessed but not in the word) and how many
wrong guesses remain, both derived purely from `(pattern, guessed_letters)`
so no interface change was needed. Previously that information was used
once (to avoid repeat guesses) and then thrown away; now it's an actual
learned input.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/bilstm-attention-features"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train

Same masked-language-model objective as the other branches: randomly mask
letters, predict the true letter at each masked position from
bidirectional context. Additionally, each training example gets a
synthetic guessed-wrong letter set (0-5 random letters absent from the
word) and a matching remaining-guesses value, so the model learns to
condition on both the board AND the guess history/budget -- not just the
board alone.

In [ ]:
!python src/train_bilstm.py --epochs 20

## Validate

Same methodology as every other branch, for a fair comparison: hold out
10% of train.txt, play full interactive games against words the model
never trained on. Compare directly against BiLSTM+attention alone to see
whether the extra features actually earn their keep.

In [ ]:
!python src/validate_bilstm.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the model
still in this session. Sandbox leaderboard checkpoint only -- per the
competition's Final Judgement policy, final hiring decisions re-run the
submitted model/notebook against a separate private word list.

250,000 words, one game at a time -- prints progress every 20,000 words
with an ETA.

In [ ]:
!python src/generate_submission_bilstm.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/bilstm_attn_feat_masker.pt", "/kaggle/working/bilstm_attn_feat_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved bilstm_attn_feat_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")